<a href="https://colab.research.google.com/github/hansjio/tsektxt-model-training/blob/master/notebooks/colab_extra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TsekTxt: extra results for the paper (no retraining, no Taglish labels)

**What this is.** One notebook for the extra numbers the paper review asked for. Nothing here retrains a model and nothing uses the Taglish set.
1. Temperature scaling (calibration): fit one number T per model on the internal **validation** split, report before/after on the internal **test** split. Predictions do not change.
2. Integrated Gradients on a sample of the internal test set (labelled *internal*), with completeness errors, and a check that temperature scaling leaves the IG word rankings unchanged.
3. Error analysis: misclassified internal test articles per error type, with their IG top words.
4. Inference time per post (T4 GPU and CPU).
5. Library versions for the reproducibility appendix.
6. Language mix of the training corpus with a real language-ID tool.
7. Re-run the comparison table (Holm column), the confidence check and the figures, then download.

**How to run.** Runtime > Change runtime type > T4 GPU. Then Runtime > Run all (about 45-75 minutes; steps 2 and 6 are the slowest). At the end a zip downloads: unzip it into your local `tsektxt-model-training/` folder (it only contains `results/` files) and tell Claude, who will check and commit them.

In [ ]:
import os
# Safe to re-run: clone only if the folder is not there yet, and do not nest folders.
if os.path.basename(os.getcwd()) != 'tsektxt-model-training':
    if not os.path.isdir('tsektxt-model-training'):
        !git clone -q https://github.com/hansjio/tsektxt-model-training.git
    %cd tsektxt-model-training
!git pull -q
!pip -q install 'transformers>=4.41,<5' scikit-learn pandas pyyaml scipy captum joblib ftfy datasets matplotlib

## 0. Rebuild the cleaned data and the 70/15/15 splits
**What it does:** recreates `data/processed/{cleaned,train,val,test}.csv` from `data/raw` with the same seed (42) used for training. About a minute.

In [ ]:
if not os.path.exists('data/processed/val.csv'):
    !python src/clean_data.py --input data/raw/combined_dataset_clean.csv
    !python src/make_splits.py
!wc -l data/processed/*.csv

## 1. Temperature scaling (post-hoc calibration)
**What it does:** runs each model on the validation and test splits (3,803 + 3,803 articles), fits T on validation only, and writes `results/calibration.json`, `results/calibration_report.json` and `results/<model>/{val,test}_logits.csv` (ids and two numbers per article, no text). It stops with an error if any prediction changes (it cannot, by construction). About 5 minutes on a T4.

**What to look for:** T above 1 means the raw scores were overconfident. ECE and NLL should go down; accuracy stays the same. `share_test_predictions_matching_test_predictions_csv` should be 1.0 or very close: it checks that this run reproduces the predictions reported in Chapter IV.

In [ ]:
!python src/calibrate.py
!python src/confidence_check.py > /dev/null && echo "confidence_check.json now has a calibrated block"

## 2. Integrated Gradients on an internal test sample (label it INTERNAL in the paper)
**What it does:** draws 300 internal test articles (stratified, seed 42), runs IG for the three models and ranks words. Output goes to `results/ig_internal/` so it never mixes with the Taglish IG files. It also reports the completeness error (how closely the attributions add up to the model output; target: within 5%). About 20-30 minutes on a T4.

**Reading it:** these words show what the models rely on for long news articles, where labels follow the source. They are descriptive, not proof of what makes a post false.

In [ ]:
import pandas as pd
t = pd.read_csv('data/processed/test.csv')
s = t.groupby('label', group_keys=False).apply(lambda g: g.sample(frac=300 / len(t), random_state=42))
s[['id', 'text', 'label']].to_csv('data/processed/ig_internal_sample.csv', index=False)
print(len(s), s.label.value_counts().to_dict())
!python src/interpret.py corpus --csv data/processed/ig_internal_sample.csv --all-posts --out-dir results/ig_internal

## 2b. Check: temperature scaling does not change the Integrated Gradients word rankings
**What it does:** on 20 internal test articles, runs IG with each model as it is, and again with the model's final layer divided by T (which is exactly softmax(logits / T)). IG attributes the logit difference, so dividing by T only scales every score by 1/T: the word order should be identical. Writes `results/ig_temperature_check.json`. About 5 minutes.

In [ ]:
import copy, json, torch, sys
sys.path.insert(0, 'src'); import interpret
from calibrate import load_temperature
s20 = pd.read_csv('data/processed/ig_internal_sample.csv').sample(20, random_state=42)
def scaled(model, T):
    m2 = copy.deepcopy(model); head = m2.classifier.out_proj if hasattr(m2.classifier, 'out_proj') else m2.classifier
    with torch.no_grad(): head.weight /= T; head.bias /= T
    return m2
out = {}
for m in interpret.NAMES:
    tok, model, _ = interpret.load(m); T, ok = load_temperature(m)
    assert ok, 'run step 1 first'
    mT = scaled(model, T); same_top10 = 0; same_pred = 0; max_ratio_dev = 0.0
    for text in s20.text.astype(str):
        a = interpret.explain(tok, model, text, n_steps=100, max_steps=100)
        b = interpret.explain(tok, mT, text, n_steps=100, max_steps=100)
        ra = [w['word'] for w in sorted(a['words'], key=lambda w: -w['attr'])][:10]
        rb = [w['word'] for w in sorted(b['words'], key=lambda w: -w['attr'])][:10]
        same_top10 += ra == rb; same_pred += a['pred_label'] == b['pred_label']
        for wa, wb in zip(a['words'], b['words']):
            if abs(wa['attr']) > 1e-6: max_ratio_dev = max(max_ratio_dev, abs(wb['attr'] * T / wa['attr'] - 1))
    out[m] = {'T': T, 'n_posts': len(s20), 'identical_top10_word_order': int(same_top10), 'identical_prediction': int(same_pred),
              'max_relative_deviation_of_attr_times_T': round(max_ratio_dev, 6)}
    print(m, out[m])
json.dump({'note': 'IG with logits/T vs raw; fixed 100 steps; internal test sample', 'models': out}, open('results/ig_temperature_check.json', 'w'), indent=2)

## 3. Error analysis: misclassified internal test articles
**What it does:** for each model, takes up to 12 credible articles it called not credible and 12 not credible articles it called credible (seed 42), runs IG on them, and saves a short excerpt (first 40 words, links and @handles removed) with the top words. Output: `results/error_analysis/`. About 10 minutes.

**Reading it:** look for patterns (short articles, satire-site names, opinion pieces, missing context). These are news articles from the public datasets, not posts by private people.

In [ ]:
import json, re, subprocess
os.makedirs('results/error_analysis', exist_ok=True)
t = pd.read_csv('data/processed/test.csv').set_index('id')
def short(x, n=40):
    x = re.sub(r'https?://\S+|www\.\S+|@\w+', '', str(x)); w = x.split()
    return ' '.join(w[:n]) + (' ...' if len(w) > n else '')
for m in ['roberta-tagalog', 'mbert', 'xlmr']:
    p = pd.read_csv(f'results/{m}/test_predictions.csv')
    fp = p[(p.y_true == 1) & (p.y_pred == 0)].sample(frac=1, random_state=42).head(12).assign(error='credible called not credible')
    fn = p[(p.y_true == 0) & (p.y_pred == 1)].sample(frac=1, random_state=42).head(12).assign(error='not credible called credible')
    e = pd.concat([fp, fn]); e['text'] = [t.loc[i, 'text'] for i in e.id]; e['label'] = e.y_true
    e[['id', 'text', 'label']].to_csv(f'data/processed/errors_{m}.csv', index=False)
    subprocess.run(['python', 'src/interpret.py', 'corpus', '--csv', f'data/processed/errors_{m}.csv', '--models', m,
                    '--all-posts', '--min-df', '1', '--out-dir', f'results/error_analysis/{m}'], check=True)
    posts = json.load(open(f'results/error_analysis/{m}/ig_attributions_{m}.json'))['posts']
    top = {}
    for r in posts:
        ws = sorted((w for w in r['words'] if w.get('attr') is not None), key=lambda w: -w['attr'])
        top[str(r['id'])] = ([w['word'] for w in ws[:5]], [w['word'] for w in ws[::-1][:5]])
    e['excerpt'] = [short(x) for x in e.text]
    e['ig_top_toward_not_credible'] = [', '.join(top.get(str(i), ([], []))[0]) for i in e.id]
    e['ig_top_toward_credible'] = [', '.join(top.get(str(i), ([], []))[1]) for i in e.id]
    e[['id', 'error', 'prob_credible', 'excerpt', 'ig_top_toward_not_credible', 'ig_top_toward_credible']].to_csv(
        f'results/error_analysis/{m}_examples.csv', index=False)
    print(m, e.error.value_counts().to_dict())

## 4. Inference time per post (Chapter III H, system evaluation)
**What it does:** times one-post-at-a-time prediction (batch size 1, like the demo) on 100 internal test articles, on the T4 and on the Colab CPU. Writes `results/inference_time.json` (median and 95th percentile in milliseconds, first 5 posts are warm-up). Integrated Gradients is not included; it is much slower.

In [ ]:
import time, statistics, torch, sys
sys.path.insert(0, 'src'); import predict
texts = pd.read_csv('data/processed/test.csv').text.astype(str).sample(100, random_state=42).tolist()
out = {'n_posts': 100, 'batch_size': 1, 'max_length': predict.MAXLEN, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'cpu_threads': torch.get_num_threads(), 'models': {}}
for m in predict.NAMES:
    tok, model = predict.load(m, 'auto'); out['models'][m] = {}
    for dev in (['cuda'] if torch.cuda.is_available() else []) + ['cpu']:
        model.to(dev); predict.DEV = dev; ms = []
        for i, x in enumerate(texts):
            if dev == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter(); predict.predict(tok, model, [x], bs=1)
            if dev == 'cuda': torch.cuda.synchronize()
            if i >= 5: ms.append((time.perf_counter() - t0) * 1000)
        ms.sort(); out['models'][m][dev] = {'median_ms': round(statistics.median(ms), 1), 'p95_ms': round(ms[int(.95 * len(ms)) - 1], 1)}
    print(m, out['models'][m])
json.dump(out, open('results/inference_time.json', 'w'), indent=2)

## 5. Library versions (reproducibility appendix)
**What it does:** records Python and library versions in `results/environment.json`. These are the versions of *this* evaluation session; training ran in an earlier Colab session.

In [ ]:
import platform, importlib.metadata as imd
pk = ['torch', 'transformers', 'datasets', 'scikit-learn', 'captum', 'numpy', 'pandas', 'scipy', 'accelerate', 'tokenizers']
def ver(p):
    try: return imd.version(p)
    except imd.PackageNotFoundError: return None
v = {p: ver(p) for p in pk}
json.dump({'python': platform.python_version(), 'packages': v, 'note': 'Colab session of the extra evaluation (calibration, IG internal sample, timing)'},
          open('results/environment.json', 'w'), indent=2); print(v)

## 6. Language mix of the training corpus (real language-ID tool)
**What it does:** uses the `lingua` language detector (offline, English vs Tagalog) on each cleaned training article in chunks of 30 words (first 300 words). An article is "english" or "tagalog" if at least 80% of its chunks are that language, otherwise "mixed". Writes `results/language_id_train.json` (counts only). About 15-30 minutes.

**Why:** the paper describes the corpus language; this checks it with a tool instead of the word-list guess in `profile_train.json` (about 86% English).

In [ ]:
!pip -q install lingua-language-detector
from lingua import Language, LanguageDetectorBuilder
from collections import Counter
det = LanguageDetectorBuilder.from_languages(Language.ENGLISH, Language.TAGALOG).build()
c = pd.read_csv('data/processed/cleaned.csv')
def lang(t):
    w = str(t).split()[:300]; ch = [' '.join(w[i:i + 30]) for i in range(0, len(w), 30)]
    r = [det.detect_language_of(x) for x in ch if len(x.split()) >= 5]
    if not r: return 'unknown'
    en = sum(x == Language.ENGLISH for x in r) / len(r); tl = sum(x == Language.TAGALOG for x in r) / len(r)
    return 'english' if en >= .8 else 'tagalog' if tl >= .8 else 'mixed'
L = [lang(x) for x in c.text]; lb = c.label.astype(str).tolist()
res = {'tool': 'lingua-language-detector (English vs Tagalog only), 30-word chunks of the first 300 words, 80% rule',
       'n': len(L), 'overall': dict(Counter(L)),
       'by_label': {k: dict(Counter(l for l, y in zip(L, lb) if y == k)) for k in ('0', '1')}, 'labels': '1 = credible, 0 = not credible'}
json.dump(res, open('results/language_id_train.json', 'w'), indent=2); print(res)

## 7. Comparison table (with Holm), figures, download
**What it does:** re-runs `compare.py` (adds the Holm-adjusted column for the 3 transformer pairs), rebuilds the figures (adds `fig_k_reliability`) and downloads a zip of the new result files.

In [ ]:
!python src/compare.py
!python src/make_figures.py
!zip -qr extra_results.zip results/calibration*.json results/*/val_logits.csv results/*/test_logits.csv results/confidence_check.json results/ig_internal results/ig_temperature_check.json results/error_analysis results/inference_time.json results/environment.json results/language_id_train.json results/comparison.md results/figures
from google.colab import files; files.download('extra_results.zip')